In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to datetime (UTC) and also keep integer min/max
for df in (df_metric, df_trace, df_log, df_error):
    df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Helper to build a compact one-row summary DataFrame for a file
def build_summary(df, file_label, name_col=None, sample_rows=5, max_uniques=20):
    row_count = int(df.shape[0])
    unique_cmdb = list(pd.Series(df.get("cmdb_id")).dropna().unique()[:max_uniques])
    if name_col is not None and name_col in df.columns:
        unique_names = list(pd.Series(df[name_col]).dropna().unique()[:max_uniques])
    else:
        # For files without a dedicated KPI/trace/log name column, use 'message' if present, else empty list
        if "message" in df.columns:
            unique_names = list(pd.Series(df["message"]).dropna().unique()[:max_uniques])
        else:
            unique_names = []
    min_ts = int(df["timestamp"].min()) if row_count > 0 else None
    max_ts = int(df["timestamp"].max()) if row_count > 0 else None
    summary = pd.DataFrame([{
        "file": file_label,
        "row_count": row_count,
        "unique_cmdb_ids": unique_cmdb,
        "unique_names": unique_names,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts
    }])
    sample = df.head(sample_rows).copy()
    # Keep sample columns compact: show original timestamp integer and parsed datetime plus first few data cols
    # (Do not expand entire raw DataFrame)
    return summary, sample

# Build summaries & samples
metric_summary, metric_sample = build_summary(df_metric, "metric.csv", name_col="kpi_name")
trace_summary, trace_sample = build_summary(df_trace, "trace.csv", name_col="trace_name")
log_summary, log_sample = build_summary(df_log, "log.csv", name_col="log_name")
error_summary, error_sample = build_summary(df_error, "error_logs.csv", name_col=None)

# Display compact results (one-row summaries and first 5-row samples for each file)
metric_summary, metric_sample, trace_summary, trace_sample, log_summary, log_sample, error_summary, error_sample

```
Out[1]:
```
Summary (plain English):

- metric.csv
  - Total rows: 1875
  - Unique cmdb_id (up to 20 shown): ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'payment', ...]
  - Unique kpi_name (up to 20 shown): ['cpu', 'latency-50', 'latency-90', 'mem', 'socket', 'worst', ...]
  - Min timestamp: 1705698600
  - Max timestamp: 1705700040
  - First 5 rows (timestamp, cmdb_id, kpi_name, value):
    1. 1705698600, carts, cpu, 1.335411e+00
    2. 1705698600, carts, latency-50, 1.299686e-02
    3. 1705698600, carts, latency-90, 2.288155e-02
    4. 1705698600, carts, mem, 2.100800e+08
    5. 1705698600, carts, socket, 10.0

- trace.csv
  - Total rows: 0
  - Unique cmdb_id: []
  - Unique trace_name: []
  - Min timestamp: None
  - Max timestamp: None
  - Sample: empty (no rows)

- log.csv
  - Total rows: 456
  - Unique cmdb_id (up to 20 shown): ['carts', 'catalogue', 'front-end', 'orders', 'payment', ...]
  - Unique log_name (up to 20 shown): ['log.error_count', 'log.row_count']
  - Min timestamp: 1705698600
  - Max timestamp: 1705700040
  - First 5 rows (timestamp, cmdb_id, log_name, value):
    1. 1705698600, carts, log.error_count, 0
    2. 1705698600, carts, log.row_count, 50
    3. 1705698600, catalogue, log.error_count, 0
    4. 1705698600, catalogue, log.row_count, 101
    5. 1705698600, front-end, log.error_count, 0

- error_logs.csv
  - Total rows: 0
  - Unique cmdb_id: []
  - Unique message: []
  - Min timestamp: None
  - Max timestamp: None
  - Sample: empty (no rows)

If you want, I can expand any of the unique-value lists (up to 20) or show more sample rows.

The original code execution output of IPython Kernel is also provided below for reference:

(         file  row_count                                    unique_cmdb_ids                                       unique_names  min_timestamp  max_timestamp
0  metric.csv       1875  [carts, carts-db, catalogue, catalogue-db, fro...  [cpu, latency-50, latency-90, mem, socket, wor...     1705698600     1705700040,     timestamp cmdb_id    kpi_name         value              timestamp_dt
0  1705698600   carts         cpu  1.335411e+00 2024-01-19 21:10:00+00:00
1  1705698600   carts  latency-50  1.299686e-02 2024-01-19 21:10:00+00:00
2  1705698600   carts  latency-90  2.288155e-02 2024-01-19 21:10:00+00:00
3  1705698600   carts         mem  2.100800e+08 2024-01-19 21:10:00+00:00
4  1705698600   carts      socket  1.000000e+01 2024-01-19 21:10:00+00:00,         file  row_count unique_cmdb_ids unique_names min_timestamp max_timestamp
0  trace.csv          0              []           []          None          None, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value, timestamp_dt]
Index: [],       file  row_count                                    unique_cmdb_ids                      unique_names  min_timestamp  max_timestamp
0  log.csv        456  [carts, catalogue, front-end, orders, payment,...  [log.error_count, log.row_count]     1705698600     1705700040,     timestamp    cmdb_id         log_name  value              timestamp_dt
0  1705698600      carts  log.error_count      0 2024-01-19 21:10:00+00:00
1  1705698600      carts    log.row_count     50 2024-01-19 21:10:00+00:00
2  1705698600  catalogue  log.error_count      0 2024-01-19 21:10:00+00:00
3  1705698600  catalogue    log.row_count    101 2024-01-19 21:10:00+00:00
4  1705698600  front-end  log.error_count      0 2024-01-19 21:10:00+00:00,              file  row_count unique_cmdb_ids unique_names min_timestamp max_timestamp
0  error_logs.csv          0              []           []          None          None, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error, timestamp_dt]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp_dt exists and is UTC datetime, and timestamp is integer
for df in (df_metric, df_log):
    if "timestamp_dt" not in df.columns:
        df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    else:
        # reparse to ensure correct dtype
        df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["timestamp"] = df["timestamp"].astype(int)

# 1) Global stats per cmdb_id & kpi_name for metric.csv
grp = df_metric.groupby(["cmdb_id", "kpi_name"])["value"]
metric_global_stats = grp.agg(
    global_count="count",
    global_p95=lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan,
    global_p5=lambda s: np.nanpercentile(s.dropna(), 5) if s.dropna().size>0 else np.nan,
    global_mean="mean",
    global_max="max"
).reset_index()

# Keep result compact and limit to top 200 rows
metric_global_stats = metric_global_stats.sort_values(["cmdb_id","kpi_name"]).reset_index(drop=True).head(200)

# 2) Filter metric.csv to incident window and compute window-level anomaly counts using global thresholds
start_ts = int(pd.to_datetime("2024-01-19 21:07:21", utc=True).timestamp())
end_ts   = int(pd.to_datetime("2024-01-19 21:37:21", utc=True).timestamp())

# Merge global thresholds back into full metric frame
metric_with_thresh = df_metric.merge(metric_global_stats, on=["cmdb_id", "kpi_name"], how="left")

# Filter to window (inclusive)
in_window = metric_with_thresh[(metric_with_thresh["timestamp"] >= start_ts) & (metric_with_thresh["timestamp"] <= end_ts)].copy()

# For each group compute window_count, num_above_p95, num_below_p5, earliest_anom_timestamp, window_max, window_min
def window_agg(g):
    wc = int(g.shape[0])
    p95 = g["global_p95"].iloc[0]
    p5  = g["global_p5"].iloc[0]
    num_above = int((g["value"] >= p95).sum()) if not np.isnan(p95) else 0
    num_below = int((g["value"] <= p5).sum()) if not np.isnan(p5) else 0
    # earliest timestamp where either breach occurs
    if num_above + num_below > 0:
        mask = (g["value"] >= p95) | (g["value"] <= p5)
        earliest = int(g.loc[mask, "timestamp"].min())
    else:
        earliest = pd.NA
    return pd.Series({
        "window_count": wc,
        "num_above_p95": num_above,
        "num_below_p5": num_below,
        "earliest_anom_timestamp": earliest,
        "window_max": g["value"].max(),
        "window_min": g["value"].min()
    })

if not in_window.empty:
    metric_window_agg = in_window.groupby(["cmdb_id", "kpi_name"]).apply(window_agg).reset_index()
    # Join global stats
    metric_window_agg = metric_window_agg.merge(metric_global_stats, on=["cmdb_id","kpi_name"], how="left")
    # Filter rows where window_count>0 and there is at least one breach
    metric_anomalies_window = metric_window_agg[
        (metric_window_agg["window_count"] > 0) &
        ((metric_window_agg["num_above_p95"] > 0) | (metric_window_agg["num_below_p5"] > 0))
    ].copy()
    # Sort by total breaches desc and limit to top 20
    metric_anomalies_window["total_breaches"] = metric_anomalies_window["num_above_p95"] + metric_anomalies_window["num_below_p5"]
    metric_anomalies_window = metric_anomalies_window.sort_values("total_breaches", ascending=False).head(20)
    # Select and order columns as requested
    metric_anomalies_window = metric_anomalies_window[
        ["cmdb_id","kpi_name","global_count","global_p95","global_p5",
         "window_count","num_above_p95","num_below_p5","earliest_anom_timestamp","window_max","window_min"]
    ]
else:
    # empty result structure
    metric_anomalies_window = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","global_count","global_p95","global_p5",
        "window_count","num_above_p95","num_below_p5","earliest_anom_timestamp","window_max","window_min"
    ])

# 3) Logs: compute global P95/P5 per cmdb_id & log_name, then window breaches
grp_log = df_log.groupby(["cmdb_id","log_name"])["value"]
log_global = grp_log.agg(
    global_p95=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan,
    global_p5=lambda s: np.nanpercentile(s.dropna(),5) if s.dropna().size>0 else np.nan
).reset_index()

# Merge thresholds and filter to window
log_with_thresh = df_log.merge(log_global, on=["cmdb_id","log_name"], how="left")
log_in_window = log_with_thresh[(log_with_thresh["timestamp"] >= start_ts) & (log_with_thresh["timestamp"] <= end_ts)].copy()

def log_window_agg(g):
    wc = int(g.shape[0])
    p95 = g["global_p95"].iloc[0]
    p5  = g["global_p5"].iloc[0]
    num_above = int((g["value"] >= p95).sum()) if not np.isnan(p95) else 0
    num_below = int((g["value"] <= p5).sum()) if not np.isnan(p5) else 0
    return pd.Series({
        "window_count": wc,
        "num_above_p95": num_above,
        "num_below_p5": num_below,
        "window_max": g["value"].max(),
        "window_min": g["value"].min()
    })

if not log_in_window.empty:
    log_window_agg_df = log_in_window.groupby(["cmdb_id","log_name"]).apply(log_window_agg).reset_index()
    log_window_agg_df = log_window_agg_df.merge(log_global, on=["cmdb_id","log_name"], how="left")
    # Determine metric type: error-like if 'error' or 'err' in name -> check above p95; else traffic-like -> check below p5
    def is_error_like(name):
        n = str(name).lower()
        return ("error" in n) or ("err" in n)
    log_window_agg_df["is_error_like"] = log_window_agg_df["log_name"].apply(is_error_like)
    # Filter rows that have window_count>0 and appropriate breaches
    log_anomalies = log_window_agg_df[
        (log_window_agg_df["window_count"] > 0) &
        (
            ((log_window_agg_df["is_error_like"]) & (log_window_agg_df["num_above_p95"] > 0)) |
            ((~log_window_agg_df["is_error_like"]) & (log_window_agg_df["num_below_p5"] > 0))
        )
    ].copy()
    # Limit to top 50 rows; sort by total breaches desc
    log_anomalies["total_breaches"] = log_anomalies["num_above_p95"] + log_anomalies["num_below_p5"]
    log_anomalies = log_anomalies.sort_values("total_breaches", ascending=False).head(50)
    # Select columns to return
    log_stats_window = log_anomalies[
        ["cmdb_id","log_name","global_p95","global_p5","window_count","num_above_p95","num_below_p5","window_max","window_min"]
    ]
else:
    log_stats_window = pd.DataFrame(columns=[
        "cmdb_id","log_name","global_p95","global_p5","window_count","num_above_p95","num_below_p5","window_max","window_min"
    ])

# Final compact outputs:
# - metric_global_stats: aggregated global stats per component-KPI (top 200)
# - metric_anomalies_window: filtered anomalies within the incident window (top 20)
# - log_stats_window: log threshold breaches within the window (top 50)
metric_global_stats, metric_anomalies_window, log_stats_window

```
Out[2]:
```
Summary (plain English):

1) metric.csv — global stats
- Total distinct component-KPI groups: 75.
- Example rows (global stats): 
  - carts / cpu: global_count=25, global_p95≈2.642, global_p5≈1.283, mean≈1.649, max≈3.391
  - carts / latency-50: global_count=25, global_p95≈0.0176, global_p5≈0.0136, mean≈0.0155
  - user-db / socket: global_count=25, global_p95=8.0, global_p5=8.0, max=8.0
- Observation: several KPIs (especially many "socket" metrics) are constant across the full file (global_p95 == global_p5 == the same value).

2) metric.csv — incident window (2024-01-19 21:07:21 — 21:37:21 UTC) anomalies
- Filtered window results include component-KPI rows that had data in the window and breached their global thresholds. Results were limited to top 20 by total breaches.
- Top entries (first examples): 
  - carts-db / socket — global_count=25, global_p95=6, global_p5=6, window_count=25, num_above_p95=25, num_below_p5=25, earliest_anom_timestamp=1705699000, window_max=6, window_min=6
  - session-db / socket — (same pattern, value 3)
  - rabbitmq / socket — (value 11)
  - rabbitmq / diskio — global_p95=0, global_p5=0, window_count=25, num_above_p95=25, num_below_p5=25, window_max=0, window_min=0
  - queue-master / socket, rabbitmq-exporter / socket, orders-db / socket, payment / socket, front-end / error, catalogue-db / socket, ...
  - catalogue / diskio — global_p95≈24246, global_p5=0, window_count=25, num_above_p95=2, num_below_p5=23, window_max≈71562, window_min=0 (shows many zeros with a few large spikes)
- Note: many groups show global_p95 == global_p5 (constant metric values). When that happens, values equal to the threshold are counted both as >=P95 and <=P5 in the counts, which produces both num_above_p95>0 and num_below_p5>0 even if the series is constant.

3) log.csv — global thresholds and window breaches
- Several log metrics were evaluated; results limited to rows with window_count>0 and breaches.
- Many services' log.error_count have global_p95=0 and global_p5=0 and show window_count=25 with breaches (counts reflect values equal to the threshold).
- Examples of log.row_count behaviour in the window:
  - carts / log.row_count — global_p95≈121.25, global_p5=25.0, window_count=24, window_max=125, window_min=25
  - queue-master / log.row_count — global_p95≈585.0, global_p5≈376.2, window_count=24, window_max=612, window_min=135
  - session-db / log.row_count — global_p95=5.0, window_count=5, window_max=5, window_min=5
- Many log.error_count rows and some log.row_count rows breached their global thresholds within the incident window.

Key takeaway
- There are multiple component-KPI and log metrics with threshold breaches inside the incident window. Some breaches are due to constant-valued metrics where global_p95 == global_p5, causing equal values to be counted as both above and below thresholds; others (e.g., catalogue diskio) show genuine low/high variance (many zeros and a few high spikes) during the window. If you want, I can (a) expand the top-20 metric anomaly table, (b) show exact earliest anomaly timestamps converted to human-readable UTC, or (c) list only truly one-sided breaches (ignore cases where p95==p5).

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id    kpi_name  global_count    global_p95     global_p5   global_mean    global_max
0     carts         cpu            25  2.641960e+00  1.283390e+00  1.649183e+00  3.391462e+00
1     carts  latency-50            25  1.760157e-02  1.360992e-02  1.552956e-02  1.787314e-02
2     carts  latency-90            25  3.899836e-02  2.309346e-02  2.882921e-02  3.943310e-02
3     carts         mem            25  2.097574e+08  2.068144e+08  2.078398e+08  2.100800e+08
4     carts      socket            25  1.059000e+01  9.183333e+00  9.896667e+00  1.060000e+01
..      ...         ...           ...           ...           ...           ...           ...
70     user    workload            25  1.936211e+01  1.696339e+01  1.841940e+01  1.942015e+01
71  user-db         cpu            25  1.037542e+00  9.211761e-01  9.844350e-01  1.054030e+00
72  user-db      diskio            25  1.535250e+06  1.366047e+06  1.449490e+06  1.582104e+06
73  user-db         mem            25  4.795443e+07  4.282985e+07  4.548319e+07  4.824809e+07
74  user-db      socket            25  8.000000e+00  8.000000e+00  8.000000e+00  8.000000e+00

[75 rows x 7 columns],               cmdb_id kpi_name  global_count    global_p95     global_p5  window_count  num_above_p95  num_below_p5  earliest_anom_timestamp    window_max    window_min
9            carts-db   socket            25  6.000000e+00  6.000000e+00          25.0           25.0          25.0             1.705699e+09  6.000000e+00  6.000000e+00
58         session-db   socket            25  3.000000e+00  3.000000e+00          25.0           25.0          25.0             1.705699e+09  3.000000e+00  3.000000e+00
51           rabbitmq   socket            25  1.100000e+01  1.100000e+01          25.0           25.0          25.0             1.705699e+09  1.100000e+01  1.100000e+01
49           rabbitmq   diskio            25  0.000000e+00  0.000000e+00          25.0           25.0          25.0             1.705699e+09  0.000000e+00  0.000000e+00
47       queue-master   socket            25  3.000000e+00  3.000000e+00          25.0           25.0          25.0             1.705699e+09  3.000000e+00  3.000000e+00
54  rabbitmq-exporter   socket            25  1.000000e+00  1.000000e+00          25.0           25.0          25.0             1.705699e+09  1.000000e+00  1.000000e+00
37          orders-db   socket            25  5.000000e+00  5.000000e+00          25.0           25.0          25.0             1.705699e+09  5.000000e+00  5.000000e+00
42            payment   socket            25  4.000000e+00  4.000000e+00          25.0           25.0          25.0             1.705699e+09  4.000000e+00  4.000000e+00
22          front-end    error            25  0.000000e+00  0.000000e+00          25.0           25.0          25.0             1.705699e+09  0.000000e+00  0.000000e+00
20       catalogue-db   socket            25  4.000000e+00  4.000000e+00          25.0           25.0          25.0             1.705699e+09  4.000000e+00  4.000000e+00
18       catalogue-db   diskio            25  0.000000e+00  0.000000e+00          25.0           25.0          25.0             1.705699e+09  0.000000e+00  0.000000e+00
74            user-db   socket            25  8.000000e+00  8.000000e+00          25.0           25.0          25.0             1.705699e+09  8.000000e+00  8.000000e+00
11          catalogue   diskio            25  2.424689e+04  0.000000e+00          25.0            2.0          23.0             1.705699e+09  7.156270e+04  0.000000e+00
15          catalogue   socket            25  8.000000e+00  6.000000e+00          25.0           12.0          12.0             1.705699e+09  8.000000e+00  6.000000e+00
69               user   socket            25  1.694000e+01  1.600000e+01          25.0            2.0          22.0             1.705699e+09  1.700000e+01  1.600000e+01
56         session-db   diskio            25  2.109905e+04  0.000000e+00          25.0            2.0          19.0             1.705699e+09  2.174128e+04  0.000000e+00
63           shipping   socket            25  6.000000e+00  5.086667e+00          25.0           15.0           2.0             1.705699e+09  6.766667e+00  5.000000e+00
26          front-end   socket            25  7.933333e+00  7.000000e+00          25.0            2.0           9.0             1.705699e+09  8.000000e+00  6.350000e+00
19       catalogue-db      mem            25  4.306805e+08  4.306780e+08          25.0            2.0           8.0             1.705699e+09  4.306818e+08  4.306780e+08
53  rabbitmq-exporter      mem            25  4.123634e+06  4.026368e+06          25.0            2.0           4.0             1.705700e+09  4.150886e+06  4.026368e+06,          cmdb_id         log_name  global_p95  global_p5  window_count  num_above_p95  num_below_p5  window_max  window_min
8         orders  log.error_count        0.00        0.0            25             25            25           0           0
10       payment  log.error_count        0.00        0.0            25             25            25           0           0
6      front-end  log.error_count        0.00        0.0            25             25            25           0           0
4      catalogue  log.error_count        0.00        0.0            25             25            25           0           0
16      shipping  log.error_count        0.00        0.0            25             25            25           0           0
18          user  log.error_count        0.00        0.0            25             25            25           0           0
12  queue-master  log.error_count        0.00        0.0            25             25            25           0           0
0          carts  log.error_count        0.00        0.0            24             24            24           0           0
2       carts-db  log.error_count        0.00        0.0            24             24            24           0           0
3       carts-db    log.row_count        1.00        1.0            24             24            23           2           1
15    session-db    log.row_count        5.00        5.0             5              5             5           5           5
14    session-db  log.error_count        0.00        0.0             5              5             5           0           0
1          carts    log.row_count      121.25       25.0            24              2             4         125          25
13  queue-master    log.row_count      585.00      376.2            25              3             2         612         135
17      shipping    log.row_count       65.00       41.8            25              3             2          68          15
19          user    log.row_count      614.00      411.4            25              3             2         641         185
11       payment    log.row_count      104.80       70.6            25              2             2         108          29
9         orders    log.row_count      129.60       83.6            25              2             2         136          30
5      catalogue    log.row_count      168.80      111.8            25              2             2         172          61
7      front-end    log.row_count     2083.80     1382.4            25              2             2        2096         730)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames/variables from previous runs: df_metric, metric_global_stats, df_log, log_global
# If not present, load them
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    metric_global_stats
except NameError:
    grp = df_metric.groupby(["cmdb_id","kpi_name"])["value"]
    metric_global_stats = grp.agg(
        global_count="count",
        global_p95=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan,
        global_p5=lambda s: np.nanpercentile(s.dropna(),5) if s.dropna().size>0 else np.nan,
        global_mean="mean",
        global_max="max"
    ).reset_index()

try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    log_global
except NameError:
    grp_log = df_log.groupby(["cmdb_id","log_name"])["value"]
    log_global = grp_log.agg(
        global_p95=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan,
        global_p5=lambda s: np.nanpercentile(s.dropna(),5) if s.dropna().size>0 else np.nan
    ).reset_index()

# Ensure timestamp/int and timestamp_dt present
for df in (df_metric, df_log):
    if "timestamp" in df.columns:
        df["timestamp"] = df["timestamp"].astype(int)
    if "timestamp_dt" not in df.columns:
        df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start_ts = int(pd.to_datetime("2024-01-19 21:07:21", utc=True).timestamp())
end_ts   = int(pd.to_datetime("2024-01-19 21:37:21", utc=True).timestamp())

# 1) Filter out constant-valued KPIs where global_p95 == global_p5
nonconst = metric_global_stats[metric_global_stats["global_p95"] != metric_global_stats["global_p5"]].copy()
nonconst_keys = set(zip(nonconst["cmdb_id"], nonconst["kpi_name"]))

# Merge thresholds into full metric data
metric_with_thresh = df_metric.merge(metric_global_stats[["cmdb_id","kpi_name","global_p95","global_p5","global_count"]],
                                     on=["cmdb_id","kpi_name"], how="left")

# Filter to incident window
metric_window = metric_with_thresh[(metric_with_thresh["timestamp"] >= start_ts) & (metric_with_thresh["timestamp"] <= end_ts)].copy()

# Keep only non-constant groups
metric_window = metric_window[metric_window.apply(lambda r: (r["cmdb_id"], r["kpi_name"]) in nonconst_keys, axis=1)]

# If empty, prepare empty result
if metric_window.empty:
    metric_nonconst_anomalies_top20 = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","global_count","global_p95","global_p5",
        "window_count","window_max","window_min","num_above_p95","num_below_p5",
        "earliest_anom_timestamp_utc","longest_consecutive_anom_minutes"
    ])
else:
    # Group and compute required metrics including longest consecutive anomaly minutes
    def compute_group_stats(g):
        g = g.sort_values("timestamp").reset_index(drop=True)
        p95 = float(g["global_p95"].iloc[0])
        p5 = float(g["global_p5"].iloc[0])
        gc = int(g["global_count"].iloc[0]) if "global_count" in g.columns else int(len(g))
        wc = int(len(g))
        num_above = int((g["value"] >= p95).sum())
        num_below = int((g["value"] <= p5).sum())
        window_max = g["value"].max()
        window_min = g["value"].min()
        # earliest anomaly timestamp (value >=p95 or <=p5)
        mask = (g["value"] >= p95) | (g["value"] <= p5)
        if mask.any():
            earliest_ts = int(g.loc[mask, "timestamp"].min())
            earliest_hr = pd.to_datetime(earliest_ts, unit="s", utc=True).isoformat()
        else:
            earliest_ts = pd.NA
            earliest_hr = pd.NA
        # compute longest consecutive anomalous minutes
        longest = 0
        current = 0
        prev_ts = None
        prev_anom = False
        for _, row in g.iterrows():
            ts = int(row["timestamp"])
            anom = (row["value"] >= p95) or (row["value"] <= p5)
            if anom:
                if prev_anom and (prev_ts is not None) and (ts - prev_ts == 60):
                    current += 1
                else:
                    current = 1
                if current > longest:
                    longest = current
                prev_anom = True
            else:
                prev_anom = False
                current = 0
            prev_ts = ts
        return pd.Series({
            "global_count": gc,
            "global_p95": p95,
            "global_p5": p5,
            "window_count": wc,
            "window_max": window_max,
            "window_min": window_min,
            "num_above_p95": num_above,
            "num_below_p5": num_below,
            "earliest_anom_timestamp_utc": earliest_hr,
            "longest_consecutive_anom_minutes": int(longest)
        })

    metric_agg = metric_window.groupby(["cmdb_id","kpi_name"]).apply(compute_group_stats).reset_index()

    # Keep only rows with window_count>0 and at least one breach
    metric_agg = metric_agg[(metric_agg["window_count"] > 0) & ((metric_agg["num_above_p95"] > 0) | (metric_agg["num_below_p5"] > 0))].copy()

    # Sort by longest_consecutive_anom_minutes desc, then by total breaches desc
    metric_agg["total_breaches"] = metric_agg["num_above_p95"] + metric_agg["num_below_p5"]
    metric_agg = metric_agg.sort_values(["longest_consecutive_anom_minutes","total_breaches"], ascending=[False,False]).head(20)

    # Select and order columns as requested
    metric_nonconst_anomalies_top20 = metric_agg[[
        "cmdb_id","kpi_name","global_count","global_p95","global_p5",
        "window_count","window_max","window_min","num_above_p95","num_below_p5",
        "earliest_anom_timestamp_utc","longest_consecutive_anom_minutes"
    ]].reset_index(drop=True)

# 2) From that candidate list, extract unique cmdb_id values (up to 20)
candidate_cmdbs = metric_nonconst_anomalies_top20["cmdb_id"].dropna().unique()[:20].tolist()

# Summarize log.csv within incident window for those cmdb_ids
# Merge global thresholds into full log data
log_with_thresh = df_log.merge(log_global, on=["cmdb_id","log_name"], how="left")

# Filter to window and candidate cmdbs
log_window_candidates = log_with_thresh[
    (log_with_thresh["timestamp"] >= start_ts) &
    (log_with_thresh["timestamp"] <= end_ts) &
    (log_with_thresh["cmdb_id"].isin(candidate_cmdbs))
].copy()

if log_window_candidates.empty:
    log_summary_candidates = pd.DataFrame(columns=[
        "cmdb_id","log_name","global_p95","global_p5","window_count","window_max","window_min","num_above_p95","num_below_p5"
    ])
else:
    def compute_log_stats(g):
        wc = int(len(g))
        gp95 = float(g["global_p95"].iloc[0]) if pd.notna(g["global_p95"].iloc[0]) else np.nan
        gp5 = float(g["global_p5"].iloc[0]) if pd.notna(g["global_p5"].iloc[0]) else np.nan
        num_above = int((g["value"] >= gp95).sum()) if not np.isnan(gp95) else 0
        num_below = int((g["value"] <= gp5).sum()) if not np.isnan(gp5) else 0
        return pd.Series({
            "global_p95": gp95,
            "global_p5": gp5,
            "window_count": wc,
            "window_max": g["value"].max(),
            "window_min": g["value"].min(),
            "num_above_p95": num_above,
            "num_below_p5": num_below
        })

    log_summary_candidates = log_window_candidates.groupby(["cmdb_id","log_name"]).apply(compute_log_stats).reset_index()

    # Keep only rows with window_count>0 (already true) and limit to top 50 rows sorted by window_count desc
    log_summary_candidates = log_summary_candidates.sort_values("window_count", ascending=False).head(50).reset_index(drop=True)

# Final compact outputs
metric_nonconst_anomalies_top20, candidate_cmdbs, log_summary_candidates

```
Out[3]:
```
Summary (plain English):

1) Metric anomalies (non-constant KPIs, top 20 by longest consecutive anomaly span)
- Top rows (columns: cmdb_id, kpi_name, global_count, global_p95, global_p5, window_count, window_max, window_min, num_above_p95, num_below_p5, earliest_anom_timestamp_UTC, longest_consecutive_anom_minutes):
  - catalogue / diskio — global_count=25, p95≈24246, p5=0, window_count=25, window_max≈71562, window_min=0, num_above=2, num_below=23, earliest=2024-01-19T21:10:00+00:00, longest_consecutive=25 min
  - user / socket — global_count=25, p95≈16.94, p5=16.0, window_count=25, window_max=17.0, window_min=16.0, num_above=2, num_below=22, earliest=2024-01-19T21:10:00+00:00, longest_consecutive=22 min
  - catalogue / socket — global_count=25, p95=8, p5=6, window_count=25, window_max=8, window_min=6, num_above=12, num_below=12, earliest=2024-01-19T21:10:00+00:00, longest_consecutive=12 min
  - shipping / socket — longest_consecutive=7 min (p95≈6.0, p5≈5.09)
  - session-db / diskio — longest_consecutive=5 min (many zeros and some spikes)
  - rabbitmq-exporter / mem — longest_consecutive=4 min
  - catalogue-db / mem — longest_consecutive=3 min
  - front-end / socket, carts / cpu, carts / latency-50, carts / latency-90, carts / mem, ... (others with shorter consecutive spans, most with 2 min)
- Interpretation: catalogue.diskio and user.socket show the longest continuous anomalous periods in the incident window. catalogue.diskio shows many zero values with a few large spikes (num_below_p5 >> num_above_p95). Several socket KPIs (catalogue, user, shipping, front-end) also show sustained anomalous spans.

2) Candidate components (unique cmdb_id from the metric candidate list, up to 20)
- ['catalogue', 'user', 'shipping', 'session-db', 'rabbitmq-exporter', 'catalogue-db', 'front-end', 'carts', 'carts-db', 'orders']

3) Log summary (within incident window) for those candidate cmdb_ids (compact)
- For these components, log metrics with window data include both log.row_count and log.error_count. Examples:
  - front-end: log.row_count (global_p95≈2083.8, p5≈1382.4, window_count=25, window_max=2096, window_min=730, num_above_p95=2, num_below_p5=2); log.error_count (global_p95=0, p5=0, window_count=25, window_max=0, num_above_p95=25, num_below_p5=25) — note: error_count thresholds are zero so equal-to-zero rows are counted as breaches.
  - catalogue: log.row_count (global_p95≈168.8, p5≈111.8, window_count=25, window_max=172, window_min=61, num_above_p95=2, num_below_p5=2); log.error_count (global_p95=0, p5=0, window_count=25, window_max=0, breaches counted similarly).
  - shipping: log.error_count (p95=0), log.row_count (p95≈65, p5≈41.8, window_count=25, window_max=68, window_min=15, num_above_p95=3, num_below_p5=2).
  - user: log.row_count (p95≈614, p5≈411.4, window_count=25, window_max=641, window_min=185, num_above_p95=3, num_below_p5=2); log.error_count (p95=0).
  - carts, carts-db, session-db, orders: similar patterns — log.row_count shows some breaches; log.error_count shows p95=0 and window rows equal to zero flagged due to threshold==0.
- Interpretation: logs for these candidate services show traffic (row_count) fluctuations and many log.error_count entries equal to zero (global thresholds of zero create flagged breaches when equal).

Overall takeaway
- The strongest metric candidates are catalogue.diskio (longest continuous anomaly, many zeros + spikes) and user.socket (very long consecutive anomalies). Several socket KPIs across services and some diskio metrics show persistent anomalies during the incident window.
- Log data for these candidate services shows traffic fluctuations (row_count) and error_count metrics that are all zero (which is ambiguous because global thresholds are zero).
- Next steps recommended: investigate catalogue.diskio and user.socket first (examine raw logs/traces around the earliest anomaly times), and treat zero-threshold error_count flags cautiously (they may be artifacts of constant-zero baselines).

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  global_count    global_p95     global_p5  window_count    window_max    window_min  num_above_p95  num_below_p5 earliest_anom_timestamp_utc  longest_consecutive_anom_minutes
0           catalogue      diskio            25  2.424689e+04  0.000000e+00            25  7.156270e+04  0.000000e+00              2            23   2024-01-19T21:10:00+00:00                                25
1                user      socket            25  1.694000e+01  1.600000e+01            25  1.700000e+01  1.600000e+01              2            22   2024-01-19T21:10:00+00:00                                22
2           catalogue      socket            25  8.000000e+00  6.000000e+00            25  8.000000e+00  6.000000e+00             12            12   2024-01-19T21:10:00+00:00                                12
3            shipping      socket            25  6.000000e+00  5.086667e+00            25  6.766667e+00  5.000000e+00             15             2   2024-01-19T21:10:00+00:00                                 7
4          session-db      diskio            25  2.109905e+04  0.000000e+00            25  2.174128e+04  0.000000e+00              2            19   2024-01-19T21:10:00+00:00                                 5
5   rabbitmq-exporter         mem            25  4.123634e+06  4.026368e+06            25  4.150886e+06  4.026368e+06              2             4   2024-01-19T21:28:00+00:00                                 4
6        catalogue-db         mem            25  4.306805e+08  4.306780e+08            25  4.306818e+08  4.306780e+08              2             8   2024-01-19T21:11:00+00:00                                 3
7           front-end      socket            25  7.933333e+00  7.000000e+00            25  8.000000e+00  6.350000e+00              2             9   2024-01-19T21:15:00+00:00                                 2
8               carts         cpu            25  2.641960e+00  1.283390e+00            25  3.391462e+00  1.147165e+00              2             2   2024-01-19T21:11:00+00:00                                 2
9               carts  latency-50            25  1.760157e-02  1.360992e-02            25  1.787314e-02  1.299686e-02              2             2   2024-01-19T21:10:00+00:00                                 2
10              carts  latency-90            25  3.899836e-02  2.309346e-02            25  3.943310e-02  2.288155e-02              2             2   2024-01-19T21:10:00+00:00                                 2
11              carts         mem            25  2.097574e+08  2.068144e+08            25  2.100800e+08  2.067569e+08              2             2   2024-01-19T21:10:00+00:00                                 2
12           carts-db      diskio            25  2.143325e+06  1.676548e+06            25  2.275432e+06  1.641380e+06              2             2   2024-01-19T21:17:00+00:00                                 2
13           carts-db         mem            25  8.274264e+07  7.718958e+07            25  8.306911e+07  7.692383e+07              2             2   2024-01-19T21:10:00+00:00                                 2
14          catalogue         cpu            25  7.924441e+01  1.769451e-01            25  7.978442e+01  1.642284e-01              2             2   2024-01-19T21:17:00+00:00                                 2
15          catalogue         mem            25  1.202739e+07  6.264259e+06            25  1.203554e+07  6.253158e+06              2             2   2024-01-19T21:13:00+00:00                                 2
16          catalogue    workload            25  4.256673e+00  3.851576e+00            25  4.395533e+00  3.783317e+00              2             2   2024-01-19T21:12:00+00:00                                 2
17          front-end         cpu            25  4.645439e+00  4.201943e+00            25  4.779556e+00  4.147124e+00              2             2   2024-01-19T21:15:00+00:00                                 2
18          front-end         mem            25  1.059944e+08  9.954942e+07            25  1.075855e+08  9.885654e+07              2             2   2024-01-19T21:10:00+00:00                                 2
19             orders  latency-50            25  3.668392e-02  3.088556e-02            25  3.682261e-02  3.035674e-02              2             2   2024-01-19T21:17:00+00:00                                 2, ['catalogue', 'user', 'shipping', 'session-db', 'rabbitmq-exporter', 'catalogue-db', 'front-end', 'carts', 'carts-db', 'orders'],        cmdb_id         log_name  global_p95  global_p5  window_count  window_max  window_min  num_above_p95  num_below_p5
0    front-end    log.row_count     2083.80     1382.4          25.0      2096.0       730.0            2.0           2.0
1    front-end  log.error_count        0.00        0.0          25.0         0.0         0.0           25.0          25.0
2    catalogue    log.row_count      168.80      111.8          25.0       172.0        61.0            2.0           2.0
3    catalogue  log.error_count        0.00        0.0          25.0         0.0         0.0           25.0          25.0
4     shipping  log.error_count        0.00        0.0          25.0         0.0         0.0           25.0          25.0
5     shipping    log.row_count       65.00       41.8          25.0        68.0        15.0            3.0           2.0
6       orders    log.row_count      129.60       83.6          25.0       136.0        30.0            2.0           2.0
7       orders  log.error_count        0.00        0.0          25.0         0.0         0.0           25.0          25.0
8         user  log.error_count        0.00        0.0          25.0         0.0         0.0           25.0          25.0
9         user    log.row_count      614.00      411.4          25.0       641.0       185.0            3.0           2.0
10    carts-db  log.error_count        0.00        0.0          24.0         0.0         0.0           24.0          24.0
11    carts-db    log.row_count        1.00        1.0          24.0         2.0         1.0           24.0          23.0
12       carts  log.error_count        0.00        0.0          24.0         0.0         0.0           24.0          24.0
13       carts    log.row_count      121.25       25.0          24.0       125.0        25.0            2.0           4.0
14  session-db    log.row_count        5.00        5.0           5.0         5.0         5.0            5.0           5.0
15  session-db  log.error_count        0.00        0.0           5.0         0.0         0.0            5.0           5.0)```
```